In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ===== Cell 1: Environment check, reproducibility, and input listing =====
import sys, platform, random, os
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import sklearn
import cv2
import PIL
import torch
import torchvision

# 1. Print versions so the environment is documented
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("Seaborn     :", sns.__version__)
print("Scikit-learn:", sklearn.__version__)
print("OpenCV      :", cv2.__version__)
print("Pillow      :", PIL.__version__)
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)

# 2. Check for GPU
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU:", torch.cuda.get_device_name(0))

# 3. Fix random seeds
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()
print("\nRandom seed set to", SEED)

# 4. Show what dataset(s) are attached to this notebook
print("\nContents of /kaggle/input:")
for name in os.listdir("/kaggle/input"):
    print("  -", name)

In [ ]:
# ===== Cell 3: Inventory of images and annotations, and matching check =====
from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/input/datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET")

# 1. One row per image file
img_rows = []
for split in ["train", "validation"]:
    for cls_dir in sorted((ROOT / split / "images").iterdir()):
        for img in cls_dir.glob("*.jpg"):
            img_rows.append({"split": split, "class": cls_dir.name,
                             "stem": img.stem, "img_path": str(img)})
img_df = pd.DataFrame(img_rows)

# 2. One row per XML file
xml_rows = []
for split in ["train", "validation"]:
    for x in (ROOT / split / "annotations").glob("*.xml"):
        xml_rows.append({"xml_split": split, "stem": x.stem, "xml_path": str(x)})
xml_df = pd.DataFrame(xml_rows)

# 3. Counts
print("Images per class and split:")
print(img_df.groupby(["class", "split"]).size().unstack())
print("\nXML files per split:")
print(xml_df.groupby("xml_split").size())

# 4. Match images to XMLs by filename stem
merged = img_df.merge(xml_df, on="stem", how="outer", indicator=True)
print("\nMatch status:")
print(merged["_merge"].value_counts())

# 5. Anything unusual?
print("\nImages with no XML:", merged[merged["_merge"] == "left_only"]["stem"].tolist())
print("XMLs with no image:", merged[merged["_merge"] == "right_only"]["stem"].tolist())
both = merged[merged["_merge"] == "both"]
print("Image and XML in different splits:")
print(both[both["split"] != both["xml_split"]][["stem", "split", "xml_split"]])
print("\nDuplicate image stems:", img_df["stem"].duplicated().sum())
print("Duplicate XML stems  :", xml_df["stem"].duplicated().sum())

In [ ]:
# ===== Cell 4: Image audit - size, mode, channels, corruption =====
from PIL import Image
import numpy as np

records, bad_files = [], []
for _, row in img_df.iterrows():
    try:
        with Image.open(row["img_path"]) as im:
            im.verify()                      # checks file integrity without decoding pixels
        with Image.open(row["img_path"]) as im:   # reopen: verify() invalidates the handle
            arr = np.array(im)
            records.append({
                "stem": row["stem"], "split": row["split"], "class": row["class"],
                "width": im.size[0], "height": im.size[1], "mode": im.mode,
                "channels": 1 if arr.ndim == 2 else arr.shape[2],
                "min": int(arr.min()), "max": int(arr.max()), "mean": float(arr.mean()),
            })
    except Exception as e:
        bad_files.append((row["img_path"], str(e)))

audit_df = pd.DataFrame(records)
print("Images audited:", len(audit_df), "| unreadable:", len(bad_files))
print("\nSize + mode combinations:")
print(audit_df.groupby(["width", "height", "mode", "channels"]).size())
print("\nPixel value range overall: min =", audit_df["min"].min(), ", max =", audit_df["max"].max())
print("\nMean pixel intensity by class:")
print(audit_df.groupby("class")["mean"].agg(["mean", "std"]).round(1))
print("\nUnreadable files:", bad_files)

In [ ]:
# ===== Cell 5: Are the 3 channels really identical (grayscale stored as RGB)? =====
max_diffs = []
for p in img_df["img_path"]:
    arr = np.array(Image.open(p)).astype(int)
    d = max(np.abs(arr[..., 0] - arr[..., 1]).max(),
            np.abs(arr[..., 0] - arr[..., 2]).max())
    max_diffs.append(d)

max_diffs = np.array(max_diffs)
print("Images where all 3 channels are exactly equal:", (max_diffs == 0).sum(), "of", len(max_diffs))
print("Largest channel difference in any image:", max_diffs.max())
print("Images with any channel difference > 0:", (max_diffs > 0).sum())

In [ ]:
# ===== Cell 6: Exact-duplicate check + class distribution chart =====
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Hash every image's raw pixel bytes
def pixel_hash(path):
    arr = np.array(Image.open(path).convert("L"))
    return hashlib.md5(arr.tobytes()).hexdigest()

img_df["hash"] = img_df["img_path"].apply(pixel_hash)

dup_groups = img_df[img_df["hash"].duplicated(keep=False)].sort_values("hash")
print("Images sharing identical pixels with another image:", len(dup_groups))
if len(dup_groups):
    print(dup_groups[["split", "class", "stem", "hash"]].to_string())

# 2. Class distribution chart (train + validation)
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=img_df, x="class", hue="split", ax=ax)
ax.set_title("NEU-DET: images per class and split")
ax.set_xlabel("Defect class"); ax.set_ylabel("Number of images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Cell 7: Representative images from each class =====
import random
random.seed(SEED)

classes = sorted(img_df["class"].unique())
n_per_class = 4
fig, axes = plt.subplots(len(classes), n_per_class, figsize=(10, 15))

for r, cls in enumerate(classes):
    paths = img_df[img_df["class"] == cls]["img_path"].tolist()
    for c, p in enumerate(random.sample(paths, n_per_class)):
        axes[r, c].imshow(Image.open(p).convert("L"), cmap="gray", vmin=0, vmax=255)
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(cls, loc="left", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Cell 8: Parse Pascal VOC XMLs into a DataFrame =====
import xml.etree.ElementTree as ET

def parse_xml(path):
    root = ET.parse(path).getroot()
    fname = root.findtext("filename")
    w = int(root.findtext("size/width"))
    h = int(root.findtext("size/height"))
    d = int(root.findtext("size/depth"))
    objs = []
    for obj in root.findall("object"):
        bb = obj.find("bndbox")
        objs.append({
            "label": obj.findtext("name"),
            "xmin": int(float(bb.findtext("xmin"))), "ymin": int(float(bb.findtext("ymin"))),
            "xmax": int(float(bb.findtext("xmax"))), "ymax": int(float(bb.findtext("ymax"))),
        })
    return fname, w, h, d, objs

ann_rows = []
for _, r in xml_df.iterrows():
    fname, w, h, d, objs = parse_xml(r["xml_path"])
    for o in objs:
        ann_rows.append({"stem": r["stem"], "xml_filename": fname,
                         "img_w": w, "img_h": h, "depth": d, **o})
ann_df = pd.DataFrame(ann_rows)

print("Total boxes:", len(ann_df), "| images with boxes:", ann_df["stem"].nunique())
print("\nBoxes per image:")
print(ann_df.groupby("stem").size().value_counts().sort_index())
print("\nLabels found in XMLs:")
print(ann_df["label"].value_counts())
print("\nImage sizes / depth declared in XMLs:")
print(ann_df.groupby(["img_w", "img_h", "depth"]).size())
print("\nFirst 5 rows:")
print(ann_df.head())

In [ ]:
# ===== Cell 9: Annotation validation =====
chk = ann_df.merge(img_df[["stem", "class", "split"]], on="stem", how="left")

# 1. Does every box's label match its image's folder class?
mism = chk[chk["label"] != chk["class"]]
print("Boxes whose label != folder class:", len(mism), "| images affected:", mism["stem"].nunique())
if len(mism):
    print(mism[["stem", "class", "label"]].head(10).to_string())

# 2. Does the <filename> inside each XML match the file it sits in?
fn_bad = chk[chk["xml_filename"] != chk["stem"] + ".jpg"]
print("\nXML <filename> mismatches:", fn_bad["stem"].nunique(), "images")

# 3. Coordinate sanity
bad_order = chk[(chk.xmin >= chk.xmax) | (chk.ymin >= chk.ymax)]
oob = chk[(chk.xmin < 0) | (chk.ymin < 0) | (chk.xmax > chk.img_w) | (chk.ymax > chk.img_h)]
print("\nBoxes with xmin>=xmax or ymin>=ymax:", len(bad_order))
print("Boxes outside the image:", len(oob))
print("Coordinate ranges -> xmin:", chk.xmin.min(), "-", chk.xmin.max(),
      "| ymin:", chk.ymin.min(), "-", chk.ymin.max(),
      "| xmax:", chk.xmax.min(), "-", chk.xmax.max(),
      "| ymax:", chk.ymax.min(), "-", chk.ymax.max())

# 4. Box size by class (fraction of image area covered)
chk["bw"] = chk.xmax - chk.xmin
chk["bh"] = chk.ymax - chk.ymin
chk["area_frac"] = (chk.bw * chk.bh) / (chk.img_w * chk.img_h)
print("\nBox size by label:")
print(chk.groupby("label")[["bw", "bh", "area_frac"]].median().round(2))
print("Smallest box (w, h):", chk.bw.min(), chk.bh.min())

# 5. Boxes per image by class
print("\nMean boxes per image by class:")
print(chk.groupby("class")["stem"].apply(lambda s: len(s) / s.nunique()).round(2))

# 6. The two special cases we flagged earlier
print("\ncrazing_240 boxes:")
print(chk[chk["stem"] == "crazing_240"][["xmin", "ymin", "xmax", "ymax", "label"]])
print("\nDuplicate pair boxes:")
print(chk[chk["stem"].isin(["patches_101", "patches_105"])][["stem", "xmin", "ymin", "xmax", "ymax"]])

In [ ]:
# ===== Cell 10: Where do label and filename mismatches come from? =====
# 1. Cross-tab: folder class (rows) vs box label (columns)
print("Folder class (rows) vs box label (columns):")
print(pd.crosstab(chk["class"], chk["label"]))

# 2. Per image: do the box labels include the folder class?
per_img = chk.groupby("stem").agg(
    folder=("class", "first"),
    split=("split", "first"),
    labels=("label", lambda s: set(s)),
)
def status(r):
    if r["labels"] == {r["folder"]}:
        return "all boxes match folder"
    if r["folder"] in r["labels"]:
        return "mixed: folder class + other"
    return "no box matches folder"
per_img["status"] = per_img.apply(status, axis=1)

print("\nImage-level status:")
print(per_img["status"].value_counts())
print("\nNon-matching images by folder class:")
print(per_img[per_img["status"] != "all boxes match folder"]["folder"].value_counts())
print("\nNon-matching images by split:")
print(per_img[per_img["status"] != "all boxes match folder"]["split"].value_counts())

# 3. XML <filename> mismatches: what are they?
fn_bad = chk[chk["xml_filename"] != chk["stem"] + ".jpg"].drop_duplicates("stem").copy()
fn_bad["same_stem"] = fn_bad["xml_filename"].str.rsplit(".", n=1).str[0] == fn_bad["stem"]
print("\nFilename mismatches:", len(fn_bad))
print("Same stem, only extension differs:", fn_bad["same_stem"].sum())
print(fn_bad[["stem", "xml_filename"]].head(8).to_string())

In [ ]:
# ===== Cell 11: Draw boxes on sample images =====
import matplotlib.patches as patches

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

colors = {"crazing": "red", "inclusion": "cyan", "patches": "yellow",
          "pitted_surface": "lime", "rolled-in_scale": "magenta", "scratches": "orange"}
print("Colour key:", colors)
path_of = img_df.set_index("stem")["img_path"].to_dict()

def draw(stems, ncols=4, title=""):
    nrows = -(-len(stems) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.3 * nrows))
    axes = np.array(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, s in zip(axes, stems):
        ax.imshow(Image.open(path_of[s]).convert("L"), cmap="gray", vmin=0, vmax=255)
        boxes = chk[chk["stem"] == s]
        for _, b in boxes.iterrows():
            ax.add_patch(patches.Rectangle((b["xmin"], b["ymin"]),
                         b["xmax"] - b["xmin"], b["ymax"] - b["ymin"],
                         fill=False, edgecolor=colors[b["label"]], linewidth=2))
        ax.set_title(f"{s}\nfolder: {boxes['class'].iloc[0]}", fontsize=9)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

# A: one image per class where every box matches the folder
ok = per_img[per_img["status"] == "all boxes match folder"]
ok_stems = [ok[ok["folder"] == c].sample(1, random_state=SEED).index[0] for c in sorted(colors)]
draw(ok_stems, ncols=3, title="Normal cases: boxes match the folder class")

# B: 8 random images whose boxes disagree with the folder
bad = per_img[per_img["status"] != "all boxes match folder"]
draw(list(bad.sample(8, random_state=SEED).index), ncols=4,
     title="Suspicious cases: some boxes disagree with the folder class")

In [ ]:
# ===== Cell 12: Build and save the master manifest =====
import os, hashlib
import numpy as np
import pandas as pd
from PIL import Image

# 1. Pixel hash (recompute only if Cell 6 was not run in this session)
if "hash" not in img_df.columns:
    img_df["hash"] = img_df["img_path"].apply(
        lambda p: hashlib.md5(np.array(Image.open(p).convert("L")).tobytes()).hexdigest())

# 2. Base table: one row per image, joined to its XML
man = img_df[["stem", "class", "split", "img_path", "hash"]].rename(columns={"split": "orig_split"})
man = man.merge(xml_df, on="stem", how="left")          # adds xml_split, xml_path

# 3. Box-level information per image
man = man.merge(chk.groupby("stem").size().rename("n_boxes"), on="stem", how="left")
extras = per_img.apply(lambda r: ",".join(sorted(r["labels"] - {r["folder"]})), axis=1)
man["extra_labels"] = man["stem"].map(extras)
man["is_mixed"] = man["extra_labels"] != ""

# 4. Duplicate flag (flag only, nothing is removed)
man = man.sort_values("stem").reset_index(drop=True)
man["dup_drop_candidate"] = man.duplicated("hash", keep="first")

# 5. Save small CSVs to the working folder
os.makedirs("/kaggle/working", exist_ok=True)
man.to_csv("/kaggle/working/manifest.csv", index=False)
ann_df.to_csv("/kaggle/working/annotations.csv", index=False)

# 6. Sanity checks
print("Manifest shape:", man.shape)
print("Mixed images:", man["is_mixed"].sum())
print("Duplicate drop candidates:", man.loc[man["dup_drop_candidate"], "stem"].tolist())
print("\nMixed images by class:")
print(man.groupby("class")["is_mixed"].sum())
print("\nExample row:")
print(man.iloc[0])

In [ ]:
# ===== Cell 12R: Rebuild manifest.csv from scratch (self-contained) =====
import os, hashlib
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image

ROOT = Path("/kaggle/input/datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET")

# 1. Images and XMLs (matched by filename stem, not by folder)
img_rows = [{"stem": p.stem, "class": c.name, "orig_split": s, "img_path": str(p)}
            for s in ["train", "validation"]
            for c in sorted((ROOT / s / "images").iterdir())
            for p in c.glob("*.jpg")]
img_df = pd.DataFrame(img_rows)
xml_df = pd.DataFrame([{"stem": x.stem, "xml_split": s, "xml_path": str(x)}
                       for s in ["train", "validation"]
                       for x in (ROOT / s / "annotations").glob("*.xml")])

# 2. Boxes
ann = []
for _, r in xml_df.iterrows():
    root = ET.parse(r["xml_path"]).getroot()
    for o in root.findall("object"):
        bb = o.find("bndbox")
        ann.append({"stem": r["stem"], "label": o.findtext("name"),
                    **{k: int(float(bb.findtext(k))) for k in ["xmin", "ymin", "xmax", "ymax"]}})
ann_df = pd.DataFrame(ann)

# 3. Pixel hash, boxes per image, mixed-defect flag, duplicate flag
img_df["hash"] = img_df["img_path"].apply(
    lambda p: hashlib.md5(np.array(Image.open(p).convert("L")).tobytes()).hexdigest())
man = img_df.merge(xml_df, on="stem", how="left")
man["n_boxes"] = man["stem"].map(ann_df.groupby("stem").size())
labels_per_img = ann_df.groupby("stem")["label"].apply(set)
man["extra_labels"] = [",".join(sorted(labels_per_img[s] - {c})) for s, c in zip(man["stem"], man["class"])]
man["is_mixed"] = man["extra_labels"] != ""
man = man.sort_values("stem").reset_index(drop=True)
man["dup_drop_candidate"] = man.duplicated("hash", keep="first")

# 4. Save and check
os.makedirs("/kaggle/working", exist_ok=True)
man.to_csv("/kaggle/working/manifest.csv", index=False)
ann_df.to_csv("/kaggle/working/annotations.csv", index=False)
print("Manifest:", man.shape, "| boxes:", len(ann_df))
print("Mixed:", int(man["is_mixed"].sum()), "| duplicate candidates:", man.loc[man["dup_drop_candidate"], "stem"].tolist())
print(man.groupby("class")["is_mixed"].sum().to_dict())

In [ ]:
# ===== Cell 13: Stratified 70/15/15 split with leakage checks =====
import os, json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
if "man" not in globals():
    man = pd.read_csv("/kaggle/working/manifest.csv")   # fallback after a session reset

df = man[~man["dup_drop_candidate"]].copy().reset_index(drop=True)   # drop patches_105
classes = sorted(df["class"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label"] = df["class"].map(class_to_idx)

trainval, test = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
train, val = train_test_split(trainval, test_size=0.15/0.85, stratify=trainval["label"], random_state=SEED)
train, val, test = train.assign(split="train"), val.assign(split="val"), test.assign(split="test")
splits_df = pd.concat([train, val, test]).sort_values("stem").reset_index(drop=True)

# Checks
print("Classes:", class_to_idx)
print("\nImages per class and split:")
print(splits_df.groupby(["class", "split"]).size().unstack()[["train", "val", "test"]])
print("\nTotals:", splits_df["split"].value_counts().to_dict())
print("Mixed images per split:", splits_df.groupby("split")["is_mixed"].sum().to_dict())
s = {k: set(splits_df[splits_df.split == k]["stem"]) for k in ["train", "val", "test"]}
h = {k: set(splits_df[splits_df.split == k]["hash"]) for k in ["train", "val", "test"]}
print("\nStem overlap train/val/test:", len(s["train"] & s["val"]), len(s["train"] & s["test"]), len(s["val"] & s["test"]))
print("Pixel-hash overlap train/val/test:", len(h["train"] & h["val"]), len(h["train"] & h["test"]), len(h["val"] & h["test"]))

splits_df[["stem", "class", "label", "split", "img_path", "is_mixed"]].to_csv("/kaggle/working/splits.csv", index=False)

In [ ]:
# ===== Cell 14: Load images, normalisation from TRAIN only, DataLoaders =====
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

def load_gray(p):
    return np.array(Image.open(p).convert("L"), dtype=np.uint8)   # 3 identical channels -> 1 channel

data = {}
for k in ["train", "val", "test"]:
    d = splits_df[splits_df.split == k]
    data[k] = (np.stack([load_gray(p) for p in d["img_path"]]), d["label"].values)

# Mean/std from the training split ONLY
train_pix = data["train"][0].astype(np.float32) / 255.0
MEAN, STD = float(train_pix.mean()), float(train_pix.std())
print(f"Train mean = {MEAN:.4f}, std = {STD:.4f}")
json.dump({"mean": MEAN, "std": STD, "seed": SEED, "classes": class_to_idx,
           "img_size": 200, "augmentation": "train only: random hflip p=.5, vflip p=.5"},
          open("/kaggle/working/preprocess_config.json", "w"), indent=2)

class DefectDS(Dataset):
    def __init__(self, imgs, labels, augment=False):
        self.imgs, self.labels, self.augment = imgs, labels, augment
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).float().div(255.0).unsqueeze(0)   # shape (1, 200, 200)
        if self.augment:
            if torch.rand(1) < 0.5: x = torch.flip(x, [2])    # horizontal flip
            if torch.rand(1) < 0.5: x = torch.flip(x, [1])    # vertical flip
        x = (x - MEAN) / STD
        return x, int(self.labels[i])

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(DefectDS(*data["train"], augment=True), batch_size=32, shuffle=True, generator=g)
val_loader   = DataLoader(DefectDS(*data["val"]),  batch_size=64, shuffle=False)
test_loader  = DataLoader(DefectDS(*data["test"]), batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("Batch shape:", tuple(xb.shape), "| labels:", yb[:10].tolist())
print(f"Normalised batch mean = {xb.mean():.3f}, std = {xb.std():.3f}")
print("Batches per epoch (train/val/test):", len(train_loader), len(val_loader), len(test_loader))

In [ ]:
# ===== Cell 15: Baseline CNN =====
import random, os
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
import torch, torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BaselineCNN(nn.Module):
    def __init__(self, n_classes=6, p_drop=0.4):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(1, 16),     # 200 -> 100
            block(16, 32),    # 100 -> 50
            block(32, 64),    # 50  -> 25
            block(64, 128),   # 25  -> 12
        )
        self.gap = nn.AdaptiveAvgPool2d(1)            # (128, 12, 12) -> (128, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_drop),
            nn.Linear(128, n_classes),                # raw scores (logits)
        )
    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

set_seed(SEED)                      # from Cell 1, so weight init is reproducible
model = BaselineCNN().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params:,}")

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params:,}")
with torch.no_grad():
    model.eval()
    out = model(torch.zeros(2, 1, 200, 200, device=device))
print("Output shape for a batch of 2:", tuple(out.shape))

In [ ]:
# ===== Cell 16: Training loop (best model chosen on VALIDATION only) =====
import os, time, copy
import torch, torch.nn as nn
import matplotlib.pyplot as plt

EPOCHS, LR = 30, 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)                       # forward pass
            loss = criterion(logits, yb)             # loss calculation
            if train:
                optimizer.zero_grad()                # clear old gradients
                loss.backward()                      # backpropagation
                optimizer.step()                     # update weights
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, best_state, best_epoch = float("inf"), None, 0
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    for k, v in zip(history, [tl, vl, ta, va]):
        history[k].append(v)
    if vl < best_val_loss:
        best_val_loss, best_epoch, best_state = vl, ep, copy.deepcopy(model.state_dict())
    print(f"Epoch {ep:2d}/{EPOCHS} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")
print(f"\nTime: {time.time()-t0:.0f}s | best epoch: {best_epoch} (val loss {best_val_loss:.3f})")

os.makedirs("/kaggle/working/models", exist_ok=True)
torch.save({"state_dict": best_state, "epoch": best_epoch, "mean": MEAN, "std": STD,
            "classes": class_to_idx}, "/kaggle/working/models/baseline_cnn_v1.pt")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ===== Cell 17: Validation metrics for baseline_cnn_v1 (best checkpoint) =====
import numpy as np, torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# 1. Load the best-epoch weights (epoch 29), not the last-epoch weights
ckpt = torch.load("/kaggle/working/models/baseline_cnn_v1.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# 2. Predict on validation
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(device))
        y_prob.append(torch.softmax(logits, dim=1).cpu())     # softmax applied here
        y_pred.append(logits.argmax(1).cpu())
        y_true.append(yb)
y_true = torch.cat(y_true).numpy()
y_pred = torch.cat(y_pred).numpy()
y_prob = torch.cat(y_prob).numpy()

names = [c for c, _ in sorted(class_to_idx.items(), key=lambda kv: kv[1])]

# 3. Overall and per-class metrics
print(f"Checkpoint epoch: {ckpt['epoch']}")
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 : {f1_score(y_true, y_pred, average='macro'):.4f}\n")
print(classification_report(y_true, y_pred, target_names=names, digits=3))

# 4. Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Baseline CNN v1 - validation confusion matrix")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

# 4b. Misclassified images
val_df = splits_df[splits_df.split == "val"].reset_index(drop=True)
wrong = np.where(y_true != y_pred)[0]
print(f"Misclassified: {len(wrong)} of {len(y_true)}")
for i in wrong:
    r = val_df.iloc[i]
    print(f"{r['stem']:<20} true={names[y_true[i]]:<16} pred={names[y_pred[i]]:<16} "
          f"conf={y_prob[i].max():.2f}  mixed={bool(r['is_mixed'])}")

In [ ]:
# ===== Cell 18: View the 5 validation errors =====
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

colors = {"crazing": "red", "inclusion": "cyan", "patches": "yellow",
          "pitted_surface": "lime", "rolled-in_scale": "magenta", "scratches": "orange"}
if "ann_df" not in globals():
    ann_df = pd.read_csv("/kaggle/working/annotations.csv")

errs = [val_df.iloc[i] for i in wrong]
fig, axes = plt.subplots(1, len(errs), figsize=(4 * len(errs), 4.4))
for ax, r, i in zip(np.atleast_1d(axes), errs, wrong):
    ax.imshow(Image.open(r["img_path"]).convert("L"), cmap="gray", vmin=0, vmax=255)
    for _, b in ann_df[ann_df["stem"] == r["stem"]].iterrows():
        ax.add_patch(patches.Rectangle((b["xmin"], b["ymin"]), b["xmax"]-b["xmin"], b["ymax"]-b["ymin"],
                     fill=False, edgecolor=colors[b["label"]], linewidth=2))
    ax.set_title(f"{r['stem']}\ntrue={names[y_true[i]]} pred={names[y_pred[i]]}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Cell 19: Baseline over 3 seeds (only the seed changes) =====
import os, time, copy, random
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def train_one(seed, epochs=30, lr=1e-3):
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    tr = DataLoader(DefectDS(*data["train"], augment=True), batch_size=32, shuffle=True, generator=g)
    va = DataLoader(DefectDS(*data["val"]), batch_size=64, shuffle=False)
    m = BaselineCNN().to(device)
    crit, opt = nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=lr)

    def evaluate():
        m.eval(); loss, ys, ps = 0.0, [], []
        with torch.no_grad():
            for xb, yb in va:
                out = m(xb.to(device)); loss += crit(out, yb.to(device)).item() * len(yb)
                ys.append(yb); ps.append(out.argmax(1).cpu())
        ys, ps = torch.cat(ys).numpy(), torch.cat(ps).numpy()
        return loss / len(ys), (ys == ps).mean(), f1_score(ys, ps, average="macro")

    best = {"loss": float("inf")}
    t0 = time.time()
    for ep in range(1, epochs + 1):
        m.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        vl, vacc, vf1 = evaluate()
        if vl < best["loss"]:
            best = {"loss": vl, "acc": vacc, "f1": vf1, "epoch": ep, "state": copy.deepcopy(m.state_dict())}
    os.makedirs("/kaggle/working/models", exist_ok=True)
    torch.save({"state_dict": best["state"], "epoch": best["epoch"], "mean": MEAN, "std": STD,
                "classes": class_to_idx}, f"/kaggle/working/models/baseline_cnn_seed{seed}.pt")
    return {"seed": seed, "best_epoch": best["epoch"], "val_loss": round(best["loss"], 4),
            "val_acc": round(best["acc"], 4), "val_macro_f1": round(best["f1"], 4),
            "last_epoch_val_f1": round(vf1, 4), "seconds": round(time.time() - t0)}

results = pd.DataFrame([train_one(s) for s in [42, 7, 123]])
print(results.to_string(index=False))
print("\nMean +/- std across seeds:")
for c in ["val_acc", "val_macro_f1"]:
    print(f"  {c}: {results[c].mean():.4f} +/- {results[c].std():.4f}  (min {results[c].min():.4f}, max {results[c].max():.4f})")
results.to_csv("/kaggle/working/exp_baseline_seeds.csv", index=False)

In [ ]:
# ===== Cell 20: ResNet18 (ImageNet-pretrained), 3 seeds, same split =====
import os, time, copy, random
import numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

IMNET_MEAN, IMNET_STD = 0.449, 0.226   # mean/std of ImageNet's 3 channels, averaged (grayscale input)

def make_resnet18(n_classes=6, pretrained=True):
    w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet18(weights=w)
    # Sum the pretrained RGB filters into 1 channel: same as feeding 3 identical channels
    old = m.conv1
    new = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new.weight.data = old.weight.data.sum(dim=1, keepdim=True)
    m.conv1 = new
    m.fc = nn.Linear(m.fc.in_features, n_classes)   # new 6-class head
    return m

class RenormDS(DefectDS):
    """Reuse DefectDS, but re-normalise with ImageNet statistics instead of the train-set ones."""
    def __getitem__(self, i):
        x, y = super().__getitem__(i)
        x = x * STD + MEAN                        # undo baseline normalisation -> [0,1]
        return (x - IMNET_MEAN) / IMNET_STD, y

def train_resnet(seed, epochs=30, lr=3e-4):
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    tr = DataLoader(RenormDS(*data["train"], augment=True), batch_size=32, shuffle=True, generator=g)
    va = DataLoader(RenormDS(*data["val"]), batch_size=64, shuffle=False)
    m = make_resnet18().to(device)
    crit, opt = nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=lr)

    def evaluate():
        m.eval(); loss, ys, ps = 0.0, [], []
        with torch.no_grad():
            for xb, yb in va:
                out = m(xb.to(device)); loss += crit(out, yb.to(device)).item() * len(yb)
                ys.append(yb); ps.append(out.argmax(1).cpu())
        ys, ps = torch.cat(ys).numpy(), torch.cat(ps).numpy()
        return loss / len(ys), (ys == ps).mean(), f1_score(ys, ps, average="macro")

    best = {"loss": float("inf")}; t0 = time.time()
    for ep in range(1, epochs + 1):
        m.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        vl, vacc, vf1 = evaluate()
        if vl < best["loss"]:
            best = {"loss": vl, "acc": vacc, "f1": vf1, "epoch": ep, "state": copy.deepcopy(m.state_dict())}
    os.makedirs("/kaggle/working/models", exist_ok=True)
    torch.save({"state_dict": best["state"], "epoch": best["epoch"], "classes": class_to_idx,
                "norm": [IMNET_MEAN, IMNET_STD]}, f"/kaggle/working/models/resnet18_seed{seed}.pt")
    return {"seed": seed, "best_epoch": best["epoch"], "val_loss": round(best["loss"], 4),
            "val_acc": round(best["acc"], 4), "val_macro_f1": round(best["f1"], 4),
            "last_epoch_val_f1": round(vf1, 4), "seconds": round(time.time() - t0)}

res_r = pd.DataFrame([train_resnet(s) for s in [42, 7, 123]])
print(res_r.to_string(index=False))
print("\nResNet18 mean +/- std:")
for c in ["val_acc", "val_macro_f1"]:
    print(f"  {c}: {res_r[c].mean():.4f} +/- {res_r[c].std():.4f}  (min {res_r[c].min():.4f}, max {res_r[c].max():.4f})")
print("\nBaseline CNN (from Cell 19): val_macro_f1 0.9914 +/- 0.0056")
res_r.to_csv("/kaggle/working/exp_resnet18_seeds.csv", index=False)

In [ ]:
# ===== Cell 21: FINAL test-set evaluation (run once) =====
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)

names = [c for c, _ in sorted(class_to_idx.items(), key=lambda kv: kv[1])]
test_df = splits_df[splits_df.split == "test"].reset_index(drop=True)
mixed = test_df["is_mixed"].values
y_true = data["test"][1]

def predict(model, ds_cls):
    dl = DataLoader(ds_cls(*data["test"]), batch_size=64, shuffle=False)
    model.eval(); probs = []
    with torch.no_grad():
        for xb, _ in dl:
            probs.append(torch.softmax(model(xb.to(device)), 1).cpu())
    return torch.cat(probs).numpy()

rows, probs_store = [], {}
for kind in ["resnet18", "baseline_cnn"]:
    for seed in [42, 7, 123]:
        if kind == "resnet18":
            m = make_resnet18(pretrained=False).to(device)
            ck = torch.load(f"/kaggle/working/models/resnet18_seed{seed}.pt", map_location=device)
            ds = RenormDS
        else:
            m = BaselineCNN().to(device)
            ck = torch.load(f"/kaggle/working/models/baseline_cnn_seed{seed}.pt", map_location=device)
            ds = DefectDS
        m.load_state_dict(ck["state_dict"])
        p = predict(m, ds); pred = p.argmax(1)
        probs_store[(kind, seed)] = p
        rows.append({"model": kind, "seed": seed,
                     "acc": accuracy_score(y_true, pred),
                     "macro_f1": f1_score(y_true, pred, average="macro"),
                     "macro_prec": precision_score(y_true, pred, average="macro"),
                     "macro_rec": recall_score(y_true, pred, average="macro"),
                     "errors": int((pred != y_true).sum()),
                     "macro_f1_clean": f1_score(y_true[~mixed], pred[~mixed], average="macro")})
res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
print("\nMean +/- std over seeds:")
print(res.groupby("model")[["acc", "macro_f1", "macro_f1_clean", "errors"]].agg(["mean", "std"]).round(4).to_string())

# Detailed report for the MAIN model, seed 42
p = probs_store[("resnet18", 42)]; pred = p.argmax(1)
print("\n=== ResNet18 seed 42: classification report (test) ===")
print(classification_report(y_true, pred, target_names=names, digits=3))
print("Confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_true, pred), index=names, columns=names).to_string())

# Misclassified test images (ResNet18 seed 42 and baseline seed 42)
for kind in ["resnet18", "baseline_cnn"]:
    pr = probs_store[(kind, 42)]; pd_ = pr.argmax(1)
    wrong = np.where(pd_ != y_true)[0]
    print(f"\n{kind} seed 42 misclassified: {len(wrong)} of {len(y_true)}")
    for i in wrong:
        r = test_df.iloc[i]
        print(f"  {r['stem']:<20} true={names[y_true[i]]:<16} pred={names[pd_[i]]:<16} "
              f"conf={pr[i].max():.2f} mixed={bool(r['is_mixed'])}")
res.to_csv("/kaggle/working/test_results.csv", index=False)

In [ ]:
import os
for f in ["manifest.csv", "splits.csv", "preprocess_config.json", "test_results.csv"]:
    print(f, os.path.exists(f"/kaggle/working/{f}"))
print(sorted(os.listdir("/kaggle/working/models")) if os.path.exists("/kaggle/working/models") else "models folder missing")

In [ ]:
# ===== Cell A: Full setup. Rebuilds manifest, split, loaders, model definitions =====
import os, json, random, hashlib
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# 1. Manifest (same logic as Cell 12R)
ROOT = Path("/kaggle/input/datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET")
img_df = pd.DataFrame([{"stem": p.stem, "class": c.name, "orig_split": s, "img_path": str(p)}
                       for s in ["train", "validation"]
                       for c in sorted((ROOT / s / "images").iterdir()) for p in c.glob("*.jpg")])
xml_df = pd.DataFrame([{"stem": x.stem, "xml_path": str(x)}
                       for s in ["train", "validation"] for x in (ROOT / s / "annotations").glob("*.xml")])
ann = []
for _, r in xml_df.iterrows():
    for o in ET.parse(r["xml_path"]).getroot().findall("object"):
        bb = o.find("bndbox")
        ann.append({"stem": r["stem"], "label": o.findtext("name"),
                    **{k: int(float(bb.findtext(k))) for k in ["xmin", "ymin", "xmax", "ymax"]}})
ann_df = pd.DataFrame(ann)
img_df["hash"] = img_df["img_path"].apply(
    lambda p: hashlib.md5(np.array(Image.open(p).convert("L")).tobytes()).hexdigest())
man = img_df.merge(xml_df, on="stem", how="left")
lab = ann_df.groupby("stem")["label"].apply(set)
man["extra_labels"] = [",".join(sorted(lab[s] - {c})) for s, c in zip(man["stem"], man["class"])]
man["is_mixed"] = man["extra_labels"] != ""
man = man.sort_values("stem").reset_index(drop=True)
man["dup_drop_candidate"] = man.duplicated("hash", keep="first")

# 2. Split (same logic and seed as Cell 13 -> identical split)
df = man[~man["dup_drop_candidate"]].copy().reset_index(drop=True)
classes = sorted(df["class"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label"] = df["class"].map(class_to_idx)
trainval, test = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
train, val = train_test_split(trainval, test_size=0.15/0.85, stratify=trainval["label"], random_state=SEED)
splits_df = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")]
                      ).sort_values("stem").reset_index(drop=True)

# 3. Data arrays, train-only normalisation, datasets
def load_gray(p): return np.array(Image.open(p).convert("L"), dtype=np.uint8)
data = {k: (np.stack([load_gray(p) for p in splits_df[splits_df.split == k]["img_path"]]),
            splits_df[splits_df.split == k]["label"].values) for k in ["train", "val", "test"]}
tp = data["train"][0].astype(np.float32) / 255.0
MEAN, STD = float(tp.mean()), float(tp.std())
IMNET_MEAN, IMNET_STD = 0.449, 0.226

class DefectDS(Dataset):
    def __init__(self, imgs, labels, augment=False):
        self.imgs, self.labels, self.augment = imgs, labels, augment
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).float().div(255.0).unsqueeze(0)
        if self.augment:
            if torch.rand(1) < 0.5: x = torch.flip(x, [2])
            if torch.rand(1) < 0.5: x = torch.flip(x, [1])
        return (x - MEAN) / STD, int(self.labels[i])

class RenormDS(DefectDS):
    def __getitem__(self, i):
        x, y = super().__getitem__(i)
        return ((x * STD + MEAN) - IMNET_MEAN) / IMNET_STD, y

# 4. Models
class BaselineCNN(nn.Module):
    def __init__(self, n_classes=6, p_drop=0.4):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(p_drop), nn.Linear(128, n_classes))
    def forward(self, x): return self.classifier(self.gap(self.features(x)))

def make_resnet18(n_classes=6, pretrained=True):
    w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet18(weights=w)
    new = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
    if pretrained: new.weight.data = m.conv1.weight.data.sum(dim=1, keepdim=True)
    m.conv1 = new
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

# 5. Save small files and print checks
os.makedirs("/kaggle/working/models", exist_ok=True)
splits_df[["stem", "class", "label", "split", "img_path", "is_mixed"]].to_csv("/kaggle/working/splits.csv", index=False)
ann_df.to_csv("/kaggle/working/annotations.csv", index=False)
json.dump({"mean": MEAN, "std": STD, "seed": SEED, "classes": class_to_idx},
          open("/kaggle/working/preprocess_config.json", "w"), indent=2)
print("Split sizes:", splits_df["split"].value_counts().to_dict())
print(f"Train mean {MEAN:.4f}, std {STD:.4f}  (earlier: 0.5035, 0.2129)")
print("Mixed per split:", splits_df.groupby("split")["is_mixed"].sum().to_dict(), "(earlier: test 16, train 80, val 27)")
print("Device:", device)

In [ ]:
# ===== Cell B: Retrain baseline (3 seeds) and ResNet18 (3 seeds); selection on VALIDATION loss only =====
import time, copy
from sklearn.metrics import f1_score

def fit(kind, seed, epochs=30):
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    ds = DefectDS if kind == "baseline_cnn" else RenormDS
    tr = DataLoader(ds(*data["train"], augment=True), batch_size=32, shuffle=True, generator=g)
    va = DataLoader(ds(*data["val"]), batch_size=64, shuffle=False)
    m = (BaselineCNN() if kind == "baseline_cnn" else make_resnet18()).to(device)
    lr = 1e-3 if kind == "baseline_cnn" else 3e-4
    crit, opt = nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=lr)
    best, t0 = {"loss": float("inf")}, time.time()
    for ep in range(1, epochs + 1):
        m.train()
        for xb, yb in tr:
            opt.zero_grad(); crit(m(xb.to(device)), yb.to(device)).backward(); opt.step()
        m.eval(); L, ys, ps = 0.0, [], []
        with torch.no_grad():
            for xb, yb in va:
                o = m(xb.to(device)); L += crit(o, yb.to(device)).item() * len(yb)
                ys.append(yb); ps.append(o.argmax(1).cpu())
        ys, ps = torch.cat(ys).numpy(), torch.cat(ps).numpy()
        if L / len(ys) < best["loss"]:
            best = {"loss": L / len(ys), "acc": (ys == ps).mean(), "f1": f1_score(ys, ps, average="macro"),
                    "epoch": ep, "state": copy.deepcopy(m.state_dict())}
    torch.save({"state_dict": best["state"], "epoch": best["epoch"], "classes": class_to_idx},
               f"/kaggle/working/models/{kind}_seed{seed}.pt")
    return {"model": kind, "seed": seed, "best_epoch": best["epoch"], "val_loss": round(best["loss"], 4),
            "val_acc": round(best["acc"], 4), "val_macro_f1": round(best["f1"], 4), "seconds": round(time.time() - t0)}

rows = [fit(k, s) for k in ["baseline_cnn", "resnet18"] for s in [42, 7, 123]]
res_val = pd.DataFrame(rows)
print(res_val.to_string(index=False))
print(res_val.groupby("model")["val_macro_f1"].agg(["mean", "std"]).round(4))
res_val.to_csv("/kaggle/working/exp_val_results.csv", index=False)
print("\nSaved:", sorted(os.listdir("/kaggle/working/models")))

In [ ]:
# ===== Cell D1: Detection dataset + sanity checks =====
import torch, numpy as np, pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt, matplotlib.patches as mpatches

W = H = 200
cls_names = sorted(class_to_idx, key=class_to_idx.get)
boxes_by_stem = {s: g for s, g in ann_df.groupby("stem")}

class DetDS(Dataset):
    def __init__(self, split, augment=False):
        self.df = splits_df[splits_df.split == split].reset_index(drop=True)
        self.augment = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = torch.from_numpy(np.array(Image.open(r["img_path"]).convert("L"))).float().div(255).unsqueeze(0).repeat(3, 1, 1)
        g = boxes_by_stem[r["stem"]]
        b = torch.tensor(g[["xmin", "ymin", "xmax", "ymax"]].values, dtype=torch.float32)
        b[:, 0] -= 1; b[:, 1] -= 1                       # 1-based -> 0-based
        b = b.clamp(0, W)
        lab = torch.tensor([class_to_idx[l] + 1 for l in g["label"]], dtype=torch.int64)   # 0 = background
        if self.augment and torch.rand(1) < 0.5:
            img = torch.flip(img, [2])
            b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], dim=1)
        return img, {"boxes": b, "labels": lab}, r["stem"]

def collate(batch): return tuple(zip(*batch))

dsets = {k: DetDS(k, augment=(k == "train")) for k in ["train", "val", "test"]}
print({k: len(v) for k, v in dsets.items()})
print("Boxes per split:", {k: int(sum(len(boxes_by_stem[s]) for s in v.df["stem"])) for k, v in dsets.items()})

# Sanity: any degenerate boxes after conversion?
bad = 0
for k, d in dsets.items():
    for s in d.df["stem"]:
        g = boxes_by_stem[s]
        bad += int(((g.xmax - g.xmin) < 2).sum() + ((g.ymax - g.ymin) < 2).sum())
print("Degenerate boxes (<2px):", bad)

# Visual check: 4 train images, boxes after conversion (no flip)
d = DetDS("train", augment=False)
fig, ax = plt.subplots(1, 4, figsize=(16, 4.4))
for a, i in zip(ax, [0, 300, 600, 900]):
    img, t, stem = d[i]
    a.imshow(img[0], cmap="gray", vmin=0, vmax=1)
    for bb, l in zip(t["boxes"], t["labels"]):
        a.add_patch(mpatches.Rectangle((bb[0], bb[1]), bb[2]-bb[0], bb[3]-bb[1], fill=False, edgecolor="red", linewidth=2))
        a.text(bb[0], bb[1]-2, cls_names[l-1], color="yellow", fontsize=7)
    a.set_title(stem, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Cell D2: Fine-tune Faster R-CNN (COCO-pretrained), validate by mAP@0.5 =====
import time, copy, os, torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def make_det():
    m = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1,
                                min_size=400, max_size=400)       # upscale 200 -> 400 so small defects get more pixels
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 7)  # 6 classes + background
    return m

@torch.no_grad()
def evaluate_map(m, ds, score_thr=0.0):
    m.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    for imgs, tgts, _ in DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collate):
        out = m([i.to(device) for i in imgs])
        preds = [{k: v.cpu() for k, v in o.items()} for o in out]
        metric.update(preds, [{"boxes": t["boxes"], "labels": t["labels"]} for t in tgts])
    return metric.compute()

set_seed(42)
g = torch.Generator().manual_seed(42)
loader = DataLoader(dsets["train"], batch_size=8, shuffle=True, collate_fn=collate, generator=g)
model_det = make_det().to(device)
params = [p for p in model_det.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=6, gamma=0.1)

EPOCHS, best = 12, {"map50": -1}
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model_det.train(); tot = 0.0
    for imgs, tgts, _ in loader:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        loss = sum(model_det(imgs, tgts).values())      # sum of the 4 detection losses
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    r = evaluate_map(model_det, dsets["val"])
    print(f"Epoch {ep:2d}/{EPOCHS} | train loss {tot/len(loader):.3f} | val mAP@0.5 {r['map_50'].item():.3f} | val mAP@[.5:.95] {r['map'].item():.3f}")
    if r["map_50"].item() > best["map50"]:
        best = {"map50": r["map_50"].item(), "epoch": ep, "state": copy.deepcopy(model_det.state_dict())}
print(f"\nTime {time.time()-t0:.0f}s | best epoch {best['epoch']} (val mAP@0.5 {best['map50']:.3f})")
os.makedirs("/kaggle/working/models", exist_ok=True)
torch.save({"state_dict": best["state"], "epoch": best["epoch"], "classes": class_to_idx},
           "/kaggle/working/models/fasterrcnn_seed42.pt")

model_det.load_state_dict(best["state"])
r = evaluate_map(model_det, dsets["val"])
print("\nPer-class val AP@[.5:.95]:", {cls_names[int(c)-1]: round(v.item(), 3) for c, v in zip(r["classes"], r["map_per_class"])})

In [ ]:
# ===== Cell D3: Detection test evaluation (epoch-7 checkpoint, run ONCE) =====
import torch, numpy as np
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import matplotlib.pyplot as plt, matplotlib.patches as mpatches

ck = torch.load("/kaggle/working/models/fasterrcnn_seed42.pt", map_location=device)
model_det.load_state_dict(ck["state_dict"]); model_det.eval()
print("Loaded epoch:", ck["epoch"])

metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
SCORE_THR, IOU_THR = 0.5, 0.5
TP = FP = FN = 0
cls_tp = np.zeros(6); cls_fp = np.zeros(6); cls_fn = np.zeros(6)
store = []
with torch.no_grad():
    for imgs, tgts, stems in DataLoader(dsets["test"], batch_size=8, shuffle=False, collate_fn=collate):
        out = model_det([i.to(device) for i in imgs])
        for o, t, s, im in zip(out, tgts, stems, imgs):
            o = {k: v.cpu() for k, v in o.items()}
            metric.update([o], [{"boxes": t["boxes"], "labels": t["labels"]}])
            keep = o["scores"] >= SCORE_THR
            pb, pl = o["boxes"][keep], o["labels"][keep]
            used = set()
            for j in range(len(pb)):                       # greedy match, same class, IoU >= 0.5
                best, bi = 0, -1
                for k in range(len(t["boxes"])):
                    if k in used or t["labels"][k] != pl[j]: continue
                    iou = box_iou(pb[j:j+1], t["boxes"][k:k+1]).item()
                    if iou > best: best, bi = iou, k
                if best >= IOU_THR:
                    used.add(bi); TP += 1; cls_tp[pl[j]-1] += 1
                else:
                    FP += 1; cls_fp[pl[j]-1] += 1
            for k in range(len(t["boxes"])):
                if k not in used: FN += 1; cls_fn[t["labels"][k]-1] += 1
            store.append((s, im, t, pb, pl, o["scores"][keep]))

r = metric.compute()
print(f"Test mAP@0.5 {r['map_50'].item():.3f} | mAP@0.75 {r['map_75'].item():.3f} | mAP@[.5:.95] {r['map'].item():.3f}")
print("Per-class AP@[.5:.95]:", {cls_names[int(c)-1]: round(v.item(), 3) for c, v in zip(r["classes"], r["map_per_class"])})
print(f"\nAt score>={SCORE_THR}, IoU>={IOU_THR}: precision {TP/max(TP+FP,1):.3f} | recall {TP/max(TP+FN,1):.3f}  (TP {TP}, FP {FP}, FN {FN})")
for i, n in enumerate(cls_names):
    print(f"  {n:<16} precision {cls_tp[i]/max(cls_tp[i]+cls_fp[i],1):.3f}  recall {cls_tp[i]/max(cls_tp[i]+cls_fn[i],1):.3f}")

fig, axes = plt.subplots(2, 4, figsize=(16, 8.4))
for ax, (s, im, t, pb, pl, sc) in zip(axes.ravel(), store[::34][:8]):
    ax.imshow(im[0], cmap="gray", vmin=0, vmax=1); ax.axis("off"); ax.set_title(s, fontsize=9)
    for b in t["boxes"]:
        ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor="lime", linewidth=2))
    for b, l, c in zip(pb, pl, sc):
        ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor="red", linewidth=2, linestyle="--"))
        ax.text(b[0], b[1]-2, f"{cls_names[l-1]} {c:.2f}", color="yellow", fontsize=7)
plt.suptitle("Green = ground truth, red dashed = prediction")
plt.tight_layout()
plt.show()

In [ ]:
# ===== Cell P1: Inspection platform (ResNet18 classifier + Faster R-CNN detector) =====
import os, numpy as np, torch, torchvision
from PIL import Image, ImageDraw
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
try:
    import gradio as gr
except ImportError:
    os.system("pip install -q gradio"); import gradio as gr

cls_names = sorted(class_to_idx, key=class_to_idx.get)

# 1. Load both models from saved checkpoints (no downloads: weights=None)
clf = make_resnet18(pretrained=False).to(device)
clf.load_state_dict(torch.load("/kaggle/working/models/resnet18_seed42.pt", map_location=device)["state_dict"])
clf.eval()
det = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, min_size=400, max_size=400)
det.roi_heads.box_predictor = FastRCNNPredictor(1024, 7)
det.load_state_dict(torch.load("/kaggle/working/models/fasterrcnn_seed42.pt", map_location=device)["state_dict"])
det.to(device).eval()

# 2. Inspection function
@torch.no_grad()
def inspect(pil_img, score_thr=0.5):
    g = pil_img.convert("L").resize((200, 200))            # models were trained on 200x200 grayscale
    x = torch.from_numpy(np.array(g)).float().div(255).unsqueeze(0).unsqueeze(0).to(device)
    probs = torch.softmax(clf((x - IMNET_MEAN) / IMNET_STD), 1)[0].cpu().numpy()
    out = det([x[0].repeat(3, 1, 1)])[0]
    canvas = g.convert("RGB").resize((400, 400)); d = ImageDraw.Draw(canvas)
    n = 0
    for b, l, s in zip(out["boxes"].cpu(), out["labels"].cpu(), out["scores"].cpu()):
        if s >= score_thr:
            b = (b * 2).tolist()                            # 200 -> 400 px for display
            d.rectangle(b, outline="red", width=2)
            d.text((b[0] + 3, b[1] + 3), f"{cls_names[int(l)-1]} {s:.2f}", fill="yellow")
            n += 1
    top = int(probs.argmax())
    verdict = (f"DEFECT: {cls_names[top]} (confidence {probs[top]:.1%}). "
               f"{n} box(es) drawn at score >= {score_thr}.")
    return canvas, {c: float(p) for c, p in zip(cls_names, probs)}, verdict

# 3. Quick check on one test image BEFORE launching the UI
r = splits_df[splits_df.split == "test"].iloc[0]
_, p, v = inspect(Image.open(r["img_path"]))
print("Test image:", r["stem"], "| true class:", r["class"]); print(v)

# 4. Launch
demo = gr.Interface(
    fn=inspect,
    inputs=[gr.Image(type="pil", label="Steel surface image"),
            gr.Slider(0.1, 0.95, value=0.5, step=0.05, label="Detection score threshold")],
    outputs=[gr.Image(label="Detected defects (prototype)"),
             gr.Label(num_top_classes=6, label="Defect class confidence (ResNet18)"),
             gr.Textbox(label="Inspection result")],
    title="Industrial Surface Defect Inspection Platform",
    description="Classifies NEU-DET steel defects into 6 types. This model has no 'normal' class, "
                "so it will always name a defect type. Box detection is a prototype.")
demo.launch(inline=True, debug=False)

In [ ]:
import matplotlib.pyplot as plt
for idx in [5, 60, 120, 200]:                      # four different test images
    r = splits_df[splits_df.split == "test"].iloc[idx]
    canvas, probs, verdict = inspect(Image.open(r["img_path"]))
    plt.figure(figsize=(5, 5)); plt.imshow(canvas); plt.axis("off")
    plt.title(f"{r['stem']} (true: {r['class']})"); plt.show()
    print(verdict)
    print({k: round(v, 3) for k, v in probs.items()}, "\n")

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/backup", exist_ok=True)
for f in ["manifest.csv","splits.csv","annotations.csv","preprocess_config.json",
          "exp_val_results.csv","test_results.csv"]:
    p = f"/kaggle/working/{f}"
    if os.path.exists(p): shutil.copy(p, "/kaggle/working/backup/")
shutil.copytree("/kaggle/working/models", "/kaggle/working/backup/models", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/project_backup", "zip", "/kaggle/working/backup")
print("Size (MB):", round(os.path.getsize("/kaggle/working/project_backup.zip")/1e6, 1))
print(sorted(os.listdir("/kaggle/working/backup/models")))

In [ ]:
import os
print(os.path.exists("/kaggle/working/project_backup.zip"), os.listdir("/kaggle/working"))

In [ ]:
from IPython.display import FileLink
FileLink("project_backup.zip", result_html_prefix="Click to download: ")

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/essential", exist_ok=True)
for f in ["resnet18_seed42.pt", "fasterrcnn_seed42.pt"]:
    shutil.copy(f"/kaggle/working/models/{f}", "/kaggle/working/essential/")
shutil.make_archive("/kaggle/working/essential_models", "zip", "/kaggle/working/essential")
print(round(os.path.getsize("/kaggle/working/essential_models.zip")/1e6, 1), "MB")